# SARCLIP -> Qwen Semantic Bridging Experiments

这个 notebook 用来验证你当前真正关心的问题：

1. `SARCLIP image` 和 `CLIP text` 原本对齐得怎么样。
2. 如果把同样的 `SARCLIP image` 特征直接拿去和 `Qwen text` 比，相似度会不会明显变坏。
3. 在**不训练或极少处理**的前提下，能不能通过桥接把 `SARCLIP image` 特征迁移到 `Qwen text` 空间，并恢复分类/检索性能。

核心实验包含：

- Baseline 1: `SARCLIP image <-> CLIP text`
- Baseline 2: `SARCLIP image <-> Qwen text`（直接硬对）
- Method 1: Procrustes
- Method 2: Prototype Interpolation
- Method 3: Whitening + Procrustes


## 实验假设与输入格式

默认你已经有：

- 一份**分类数据集清单**，至少包含：`feature_path`, `label`, `split`
- 每个 `feature_path` 对应一个 `torch.Tensor`，是 **SARCLIP image encoder** 输出的图像特征
- 一组类别名，例如：`airport`, `bridge`, `harbor`, `ship`, `stadium` ...

如果你现在只有 caption 数据，也可以先把 `label` 换成你定义好的类别，或者把 prototype bank 做成更丰富的语义短语库。

In [ ]:
# 如果缺包，可以先执行这一行
# %pip install -q open_clip_torch transformers pandas scikit-learn seaborn matplotlib tqdm

import json
import math
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
import open_clip

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


In [ ]:
CFG = {
    # -----------------------------
    # 数据
    # -----------------------------
    'manifest_path': '/path/to/your/classification_manifest.csv',
    'feature_column': 'feature_path',
    'label_column': 'label',
    'split_column': 'split',
    'eval_split': 'test',

    # -----------------------------
    # 模型
    # -----------------------------
    'clip_model_name': 'ViT-B-32',
    'clip_pretrained': 'openai',
    # 建议这里优先放 text-only Qwen 模型路径。
    # 如果你确实要用别的 Qwen 变体，也可以改这里。
    'qwen_model_name_or_path': '/path/to/your/qwen_text_model',
    'torch_dtype': torch.float16 if torch.cuda.is_available() else torch.float32,

    # -----------------------------
    # 类别与 prompt
    # -----------------------------
    'class_names': [
        'airport',
        'bridge',
        'harbor',
        'ship',
        'stadium',
        'storage tank',
        'residential area',
        'farmland',
        'forest',
        'river',
        'road',
    ],
    'prompt_templates': [
        'a SAR image of {}',
        'a remote sensing SAR image showing {}',
        'an aerial SAR view of {}',
        'a SAR scene containing {}',
        'a concise SAR description of {}',
    ],
    # 更丰富的 prototype 用于 bridge 拟合与插值。
    'prototype_templates': [
        'a SAR image of {}',
        'a remote sensing SAR image showing {}',
        'an aerial SAR view of {}',
        'a SAR scene containing {}',
        'a satellite SAR observation of {}',
    ],
    'prototype_extra_phrases': [
        'urban area',
        'dense buildings',
        'industrial area',
        'runway',
        'port area',
        'coastline',
        'river crossing',
        'road network',
        'agricultural fields',
        'forest region',
    ],

    # -----------------------------
    # 批大小与 bridge 超参
    # -----------------------------
    'batch_size_text': 32,
    'topk_interp': 8,
    'temperature_interp': 20.0,
    'eps': 1e-6,
}

CFG


## 期望的数据清单格式

建议准备一个 `csv` 或 `jsonl`，至少包含以下列：

- `feature_path`: 指向单个 `.pt` 特征文件
- `label`: 类别名
- `split`: 例如 `train / val / test`

例如：

| feature_path | label | split |
|---|---|---|
| `/mnt/data/.../img_0001.pt` | `airport` | `test` |
| `/mnt/data/.../img_0002.pt` | `bridge` | `test` |


In [ ]:
def load_manifest(path: str, eval_split: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    elif path.suffix.lower() in {'.jsonl', '.json'}:
        if path.suffix.lower() == '.jsonl':
            rows = [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
            df = pd.DataFrame(rows)
        else:
            df = pd.DataFrame(json.loads(path.read_text(encoding='utf-8')))
    else:
        raise ValueError(f'Unsupported manifest suffix: {path.suffix}')

    required = {CFG['feature_column'], CFG['label_column']}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    split_col = CFG['split_column']
    if split_col in df.columns:
        df = df[df[split_col] == eval_split].copy()

    df = df.reset_index(drop=True)
    print('Loaded rows =', len(df))
    return df


def load_feature_tensor(path: str) -> torch.Tensor:
    x = torch.load(path, map_location='cpu')
    if not isinstance(x, torch.Tensor):
        raise TypeError(f'Feature is not a tensor: {path}')
    x = x.float()
    if x.ndim > 1:
        # 允许你直接放 patch tokens；默认做 mean pool 变成单向量。
        x = x.mean(dim=0)
    return x


def load_image_features(df: pd.DataFrame) -> Tuple[torch.Tensor, List[str]]:
    feats = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading image features'):
        feat = load_feature_tensor(str(row[CFG['feature_column']]))
        feats.append(feat)
        labels.append(str(row[CFG['label_column']]))

    X = torch.stack(feats, dim=0)
    X = F.normalize(X, dim=-1)
    return X, labels


In [ ]:
def load_clip_text_model(model_name: str, pretrained: str):
    model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=DEVICE)
    tokenizer = open_clip.get_tokenizer(model_name)
    model.eval()
    return model, tokenizer


def load_qwen_text_model(model_name_or_path: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        model = AutoModel.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            torch_dtype=CFG['torch_dtype'],
            device_map='auto' if torch.cuda.is_available() else None,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            torch_dtype=CFG['torch_dtype'],
            device_map='auto' if torch.cuda.is_available() else None,
        )
    model.eval()
    return model, tokenizer


def last_token_pool(hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    last_idx = attention_mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0), device=hidden.device), last_idx]


In [ ]:
def encode_clip_texts(texts: Sequence[str], clip_model, clip_tokenizer, batch_size: int = 32) -> torch.Tensor:
    outs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding CLIP text'):
        batch = list(texts[i:i + batch_size])
        toks = clip_tokenizer(batch).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.encode_text(toks)
            emb = F.normalize(emb.float(), dim=-1)
        outs.append(emb.cpu())
    return torch.cat(outs, dim=0)


def encode_qwen_texts(texts: Sequence[str], qwen_model, qwen_tokenizer, batch_size: int = 32) -> torch.Tensor:
    outs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding Qwen text'):
        batch = list(texts[i:i + batch_size])
        tok = qwen_tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors='pt',
        )
        tok = {k: v.to(qwen_model.device if hasattr(qwen_model, 'device') else DEVICE) for k, v in tok.items()}
        with torch.no_grad():
            out = qwen_model(**tok, output_hidden_states=True, return_dict=True)
            hidden = out.last_hidden_state if hasattr(out, 'last_hidden_state') else out.hidden_states[-1]
            emb = last_token_pool(hidden.float(), tok['attention_mask'])
            emb = F.normalize(emb, dim=-1)
        outs.append(emb.cpu())
    return torch.cat(outs, dim=0)


def make_prompt_ensemble(class_names: Sequence[str], templates: Sequence[str]) -> List[List[str]]:
    return [[tpl.format(name) for tpl in templates] for name in class_names]


def encode_prompt_ensemble(ensemble_prompts: Sequence[Sequence[str]], encode_fn) -> torch.Tensor:
    class_vecs = []
    for prompts in tqdm(ensemble_prompts, desc='Prompt ensemble'):
        emb = encode_fn(list(prompts))
        emb = emb.mean(dim=0, keepdim=True)
        emb = F.normalize(emb, dim=-1)
        class_vecs.append(emb)
    return torch.cat(class_vecs, dim=0)


def build_prototype_texts(class_names: Sequence[str], templates: Sequence[str], extra_phrases: Sequence[str]) -> List[str]:
    texts = []
    for name in class_names:
        texts.extend([tpl.format(name) for tpl in templates])
    texts.extend(extra_phrases)
    # 去重但保序
    seen = set()
    uniq = []
    for t in texts:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return uniq


In [ ]:
def cosine_scores(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    a = F.normalize(a, dim=-1)
    b = F.normalize(b, dim=-1)
    return a @ b.T


def topk_accuracy(scores: torch.Tensor, gt_idx: torch.Tensor, ks=(1, 5)) -> Dict[str, float]:
    out = {}
    for k in ks:
        pred = scores.topk(k=min(k, scores.size(1)), dim=1).indices
        hit = (pred == gt_idx.unsqueeze(1)).any(dim=1).float().mean().item()
        out[f'top{k}'] = hit
    return out


def evaluate_closed_set(image_embs: torch.Tensor, text_embs: torch.Tensor, labels: List[str], class_names: Sequence[str]) -> Dict[str, object]:
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    gt_idx = torch.tensor([class_to_idx[x] for x in labels], dtype=torch.long)
    scores = cosine_scores(image_embs, text_embs)
    pred_idx = scores.argmax(dim=1)
    metrics = topk_accuracy(scores, gt_idx, ks=(1, 5))
    cm = confusion_matrix(gt_idx.numpy(), pred_idx.numpy(), labels=np.arange(len(class_names)))
    return {
        'scores': scores,
        'pred_idx': pred_idx,
        'gt_idx': gt_idx,
        'top1': metrics['top1'],
        'top5': metrics['top5'],
        'confusion_matrix': cm,
    }


def plot_confusion(cm: np.ndarray, class_names: Sequence[str], title: str):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.title(title)
    plt.xlabel('Pred')
    plt.ylabel('GT')
    plt.tight_layout()


In [ ]:
def fit_procrustes(C: torch.Tensor, Q: torch.Tensor, center: bool = True):
    C = F.normalize(C, dim=-1)
    Q = F.normalize(Q, dim=-1)
    c_mean = C.mean(dim=0, keepdim=True) if center else torch.zeros_like(C[:1])
    q_mean = Q.mean(dim=0, keepdim=True) if center else torch.zeros_like(Q[:1])
    C0 = C - c_mean
    Q0 = Q - q_mean
    M = C0.T @ Q0
    U, _, Vt = torch.linalg.svd(M, full_matrices=False)
    W = U @ Vt
    return {'W': W, 'c_mean': c_mean, 'q_mean': q_mean}


def apply_procrustes(X: torch.Tensor, state: Dict[str, torch.Tensor]) -> torch.Tensor:
    X0 = X - state['c_mean']
    Y = X0 @ state['W'] + state['q_mean']
    return F.normalize(Y, dim=-1)


def fit_whitening_procrustes(C: torch.Tensor, Q: torch.Tensor, eps: float = 1e-6):
    C = F.normalize(C, dim=-1)
    Q = F.normalize(Q, dim=-1)
    c_mean = C.mean(dim=0, keepdim=True)
    q_mean = Q.mean(dim=0, keepdim=True)
    C0 = C - c_mean
    Q0 = Q - q_mean

    cov_c = (C0.T @ C0) / max(C0.size(0) - 1, 1)
    cov_q = (Q0.T @ Q0) / max(Q0.size(0) - 1, 1)

    evals_c, evecs_c = torch.linalg.eigh(cov_c)
    evals_q, evecs_q = torch.linalg.eigh(cov_q)

    Wc = evecs_c @ torch.diag(torch.rsqrt(torch.clamp(evals_c, min=eps))) @ evecs_c.T
    Cc = evecs_q @ torch.diag(torch.sqrt(torch.clamp(evals_q, min=eps))) @ evecs_q.T

    C_white = C0 @ Wc
    Q_white = Q0 @ Wc  # 这里只是为了求旋转，后面再 color 到 Q 分布
    M = C_white.T @ Q_white
    U, _, Vt = torch.linalg.svd(M, full_matrices=False)
    R = U @ Vt

    return {
        'c_mean': c_mean,
        'q_mean': q_mean,
        'Wc': Wc,
        'Cq': Cc,
        'R': R,
    }


def apply_whitening_procrustes(X: torch.Tensor, state: Dict[str, torch.Tensor]) -> torch.Tensor:
    X0 = X - state['c_mean']
    Xw = X0 @ state['Wc']
    Xr = Xw @ state['R']
    Xc = Xr @ state['Cq'] + state['q_mean']
    return F.normalize(Xc, dim=-1)


def prototype_interpolate(image_embs: torch.Tensor, clip_proto: torch.Tensor, qwen_proto: torch.Tensor, topk: int = 8, tau: float = 20.0) -> torch.Tensor:
    sims = cosine_scores(image_embs, clip_proto)
    top_vals, top_idx = sims.topk(k=min(topk, sims.size(1)), dim=1)
    weights = F.softmax(tau * top_vals, dim=1)
    gathered = qwen_proto[top_idx]  # [B, K, D]
    mixed = (weights.unsqueeze(-1) * gathered).sum(dim=1)
    return F.normalize(mixed, dim=-1)


## Step 1: 加载数据与模型

先把 `CFG['manifest_path']` 和 `CFG['qwen_model_name_or_path']` 改成你自己的路径。

In [ ]:
df = load_manifest(CFG['manifest_path'], CFG['eval_split'])
image_embs, labels = load_image_features(df)
print('image_embs =', tuple(image_embs.shape))
print('num labels =', len(labels))

clip_model, clip_tokenizer = load_clip_text_model(CFG['clip_model_name'], CFG['clip_pretrained'])
qwen_model, qwen_tokenizer = load_qwen_text_model(CFG['qwen_model_name_or_path'])


## Step 2: 构造类别文本原型与 bridge prototype bank

In [ ]:
class_names = CFG['class_names']

clip_class_prompts = make_prompt_ensemble(class_names, CFG['prompt_templates'])
qwen_class_prompts = make_prompt_ensemble(class_names, CFG['prompt_templates'])

clip_class_text = encode_prompt_ensemble(
    clip_class_prompts,
    lambda texts: encode_clip_texts(texts, clip_model, clip_tokenizer, CFG['batch_size_text'])
)
qwen_class_text = encode_prompt_ensemble(
    qwen_class_prompts,
    lambda texts: encode_qwen_texts(texts, qwen_model, qwen_tokenizer, CFG['batch_size_text'])
)

prototype_texts = build_prototype_texts(class_names, CFG['prototype_templates'], CFG['prototype_extra_phrases'])
clip_proto = encode_clip_texts(prototype_texts, clip_model, clip_tokenizer, CFG['batch_size_text'])
qwen_proto = encode_qwen_texts(prototype_texts, qwen_model, qwen_tokenizer, CFG['batch_size_text'])

print('clip_class_text =', tuple(clip_class_text.shape))
print('qwen_class_text =', tuple(qwen_class_text.shape))
print('prototype bank =', len(prototype_texts))


## Step 3: 运行 5 组实验

- `baseline_clip`: 原始 SARCLIP image <-> CLIP text
- `baseline_qwen_direct`: SARCLIP image <-> Qwen text（直接硬对）
- `procrustes`
- `prototype_interp`
- `whitening_procrustes`


In [ ]:
results = {}

# Baseline 1
results['baseline_clip'] = evaluate_closed_set(image_embs, clip_class_text, labels, class_names)

# Baseline 2
results['baseline_qwen_direct'] = evaluate_closed_set(image_embs, qwen_class_text, labels, class_names)

# Method 1: Procrustes
proc_state = fit_procrustes(clip_proto, qwen_proto, center=True)
image_proc = apply_procrustes(image_embs, proc_state)
results['procrustes'] = evaluate_closed_set(image_proc, qwen_class_text, labels, class_names)

# Method 2: Prototype Interpolation
image_interp = prototype_interpolate(
    image_embs,
    clip_proto,
    qwen_proto,
    topk=CFG['topk_interp'],
    tau=CFG['temperature_interp'],
)
results['prototype_interp'] = evaluate_closed_set(image_interp, qwen_class_text, labels, class_names)

# Method 3: Whitening + Procrustes
wp_state = fit_whitening_procrustes(clip_proto, qwen_proto, eps=CFG['eps'])
image_wp = apply_whitening_procrustes(image_embs, wp_state)
results['whitening_procrustes'] = evaluate_closed_set(image_wp, qwen_class_text, labels, class_names)

summary = []
for name, r in results.items():
    summary.append({'method': name, 'top1': r['top1'], 'top5': r['top5']})

summary_df = pd.DataFrame(summary).sort_values('top1', ascending=False).reset_index(drop=True)
summary_df


In [ ]:
plt.figure(figsize=(10, 5))
plot_df = summary_df.melt(id_vars='method', value_vars=['top1', 'top5'], var_name='metric', value_name='score')
sns.barplot(data=plot_df, x='method', y='score', hue='metric')
plt.xticks(rotation=20, ha='right')
plt.title('SARCLIP -> Qwen Bridging Results')
plt.tight_layout()
plt.show()


In [ ]:
# 可选：查看某个方法的混淆矩阵
method_name = 'procrustes'  # 改成 baseline_clip / baseline_qwen_direct / prototype_interp / whitening_procrustes
plot_confusion(results[method_name]['confusion_matrix'], class_names, f'Confusion Matrix: {method_name}')


## 如何解释结果

你主要看这几条：

1. `baseline_clip` 是你的原始参考线。
2. `baseline_qwen_direct` 如果明显掉很多，说明 **空间不一致** 这个问题真实存在。
3. `procrustes > baseline_qwen_direct`，说明存在可解析的**全局线性桥接**。
4. `prototype_interp > procrustes`，说明两个空间更像是**局部语义可对应**，而不是统一的全局旋转。
5. `whitening_procrustes > procrustes`，说明差异不只是旋转，还包含**分布尺度 / 协方差结构**差异。

如果你后面要补检索实验，也很简单：

- 用同样的 `cosine_scores(image_embs, text_embs)`
- 把类别原型换成更细粒度文本库或完整 caption 库
- 再算 `Recall@K` 就可以


## 和你当前实验怎么对接

建议你这样落地：

1. 先准备一个闭集分类数据清单，确保每张图像都有离散标签。
2. 先只跑 `baseline_clip / baseline_qwen_direct / procrustes` 三组。
3. 如果 `baseline_qwen_direct` 很差，而 `procrustes` 能拉回来，说明桥接有意义。
4. 再补 `prototype_interp` 和 `whitening_procrustes`，判断是全局线性更合适，还是局部插值更合适。
5. 最后再把最优桥接方法迁移到 caption / retrieval 场景里。

如果你愿意，下一步我们可以直接继续做两件事之一：

- 把这个 notebook 进一步改成**适配你服务器上真实路径**的版本
- 我再给你补一个**配套的 manifest 生成脚本**，把你现在的 `.pt + label` 数据整理成 notebook 可直接读取的格式
